# Demo 2: Phân mảnh Văn bản Đệ quy & Độ chồng lấp (Recursive Character Splitting & Overlap)

## 1. Lý thuyết & Khái niệm
Khi có một đoạn văn bản sạch dài (ví dụ: Item 7 dài 10,000 từ), chúng ta không thể nạp trực tiếp toàn bộ vào cơ sở dữ liệu vector hoặc LLM. Chúng ta phải thực hiện **Chunking (Phân mảnh)**.

### Thuật toán Phân mảnh Đệ quy (Recursive Character Splitting):
Đây là thuật toán phân tách văn bản chuẩn mực nhất cho văn bản tài liệu dạng văn xuôi. Thay vì cắt văn bản một cách cứng nhắc ở vị trí bất kỳ (dễ làm đứt đôi một câu hoặc một từ), thuật toán này sử dụng một danh sách các ký tự phân tách có độ ưu tiên giảm dần:
1. `\n\n` (Phân tách giữa các đoạn văn - Ưu tiên cao nhất để giữ nguyên vẹn đoạn văn)
2. `\n` (Phân tách giữa các dòng)
3. ` ` (Khoảng trắng - Phân tách giữa các từ)
4. `""` (Ký tự rỗng - Cắt từng chữ cái, giải pháp bất đắc dĩ)

**Cơ chế đệ quy:** Thuật toán thử cắt văn bản bằng dấu `\n\n`. Nếu các mảnh thu được có kích thước nhỏ hơn `chunk_size`, nó sẽ giữ nguyên. Nếu mảnh nào lớn hơn `chunk_size`, nó sẽ đệ quy gọi lại chính nó trên mảnh đó bằng ký tự tiếp theo trong danh sách (dấu `\n`), rồi đến khoảng trắng, cho đến khi mọi mảnh đều nằm trong giới hạn.

### Khái niệm Overlap (Độ chồng lấp):
Khi cắt văn bản, thông tin ở ranh giới cắt dễ bị mất ngữ cảnh (ví dụ: câu trước bị cắt sang Chunk 1, câu sau bị đẩy sang Chunk 2). 
Để giải quyết, **Chunk Overlap** định nghĩa một khoảng văn bản chồng chéo. Phần cuối của Chunk 1 sẽ được lặp lại ở đầu của Chunk 2. Điều này duy trì tính liên tục của ngữ cảnh (Context Continuity).

### Quy trình Dữ liệu (Data Flow):
* **Input:** Chuỗi văn bản sạch từ bước Parsing (ví dụ: Text sạch của Item 7).
* **Output:** Danh sách các đoạn văn nhỏ (mảnh/chunks) có độ dài tối đa xác định, kèm theo metadata đầy đủ.
* **Mục đích của Output:** Các chunks này sẽ được ghi vào file lưu trữ cuối cùng (`data/processed/documents.jsonl`). Sau đó, các file tạo Index (BM25, FAISS Vector) sẽ đọc file JSONL này để tiến hành mã hóa và lập chỉ mục tìm kiếm.

## 2. Khởi tạo đoạn văn bản mẫu (Input)
Chúng ta chuẩn bị một đoạn văn bản mẫu về tình hình kinh doanh của Amazon để thực hiện phân mảnh.

In [ ]:
sample_text = (
    "We are Amazon.com, Inc. and we focus on customer obsession. During 2023, our net sales "
    "increased 12% to $574.8 billion compared to $514.0 billion in 2022. This performance was "
    "primarily driven by AWS cloud segment growth and net sales in retail.\n\n"
    "Our operating income increased to $36.9 billion in 2023, compared to $12.2 billion in 2022. "
    "We expect the macroeconomic environment to continue to impact our operational results. "
    "Therefore, we make significant investments in our software and satellite networks to secure "
    "our long-term growth opportunities.\n\n"
    "Our capital expenditures (specifically purchases of property and equipment) were $52,729 million "
    "in fiscal year 2023, down from $63,645 million in fiscal year 2022."
)
print("=== INPUT (Văn bản cần phân mảnh) ===")
print(sample_text)

## 3. Cài đặt Mô phỏng Thuật toán Phân mảnh Đệ quy
Chúng ta sẽ tự viết thuật toán phân mảnh đệ quy bằng Python để in log chi tiết từng bước hoạt động của thuật toán.

In [ ]:
def recursive_split(text, separators, max_size, overlap):
    """
    Hàm phân mảnh đệ quy mô phỏng theo logic của LangChain
    """
    # Nếu văn bản đã đủ nhỏ, trả về luôn
    if len(text) <= max_size:
        return [text]
    
    # Nếu không còn ký tự phân tách nào, buộc phải cắt cứng theo độ dài
    if not separators:
        return [text[i:i+max_size] for i in range(0, len(text), max_size - overlap)]
    
    # Lấy ký tự phân tách hiện tại có độ ưu tiên cao nhất
    separator = separators[0]
    next_separators = separators[1:]
    
    # Tách văn bản thành các đoạn nhỏ hơn
    print(f"[Log] Thử phân tách bằng ký tự: '{repr(separator)}'")
    splits = text.split(separator)
    
    chunks = []
    current_chunk = ""
    
    for split in splits:
        # Nếu phần tử tách ra quá lớn, đệ quy gọi tiếp bằng ký tự phân tách tiếp theo
        if len(split) > max_size:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""
            
            print(f"  -> Phát hiện đoạn dài ({len(split)} ký tự) vượt ngưỡng {max_size}. Chuyển đệ quy...")
            recursive_chunks = recursive_split(split, next_separators, max_size, overlap)
            chunks.extend(recursive_chunks)
        else:
            # Gộp các mảnh nhỏ lại cho đến khi gần chạm giới hạn max_size
            potential_chunk = current_chunk + (separator if current_chunk else "") + split
            if len(potential_chunk) <= max_size:
                current_chunk = potential_chunk
            else:
                # Đã chạm giới hạn, lưu chunk hiện tại
                if current_chunk:
                    chunks.append(current_chunk)
                # Khởi tạo chunk mới, giữ lại phần Overlap từ cuối chunk trước
                overlap_text = current_chunk[-overlap:] if current_chunk and overlap > 0 else ""
                current_chunk = overlap_text + (separator if overlap_text else "") + split
                
    if current_chunk:
        chunks.append(current_chunk)
        
    return chunks

# Thiết lập tham số phân mảnh
CHUNK_SIZE = 250
CHUNK_OVERLAP = 50
SEPARATORS = ["\n\n", "\n", " ", ""]

print(f"=== THIẾT LẬP THAM SỐ ===")
print(f"Chunk Size (Độ dài tối đa): {CHUNK_SIZE} ký tự")
print(f"Chunk Overlap (Độ rộng chồng lấn): {CHUNK_OVERLAP} ký tự\n")

final_chunks = recursive_split(sample_text, SEPARATORS, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"\n[SUCCESS] Đã chia thành {len(final_chunks)} chunks!")

## 4. Trực quan hóa kết quả phân mảnh và độ chồng lấp (Overlap)
Chúng sẽ hiển thị nguyên văn từng chunk để xem các ranh giới cắt và phần văn bản trùng lặp.

In [ ]:
print("=== KẾT QUẢ CÁC CHUNKS SAU KHI CẮT ===")
for idx, chunk in enumerate(final_chunks):
    print(f"\n----------------- CHUNK {idx+1} (Độ dài: {len(chunk)} ký tự) -----------------")
    print(chunk)
    
# Trực quan hóa Overlap giữa Chunk 1 và Chunk 2
if len(final_chunks) >= 2:
    c1 = final_chunks[0]
    c2 = final_chunks[1]
    
    # Tìm phần chuỗi trùng nhau ở cuối c1 và đầu c2
    overlap_found = ""
    for i in range(1, len(c2)):
        suffix = c1[-i:]
        if c2.startswith(suffix):
            overlap_found = suffix
            
    print("\n" + "="*60)
    print("TRỰC QUAN HÓA OVERLAP (GIAO THOA NGỮ CẢNH)")
    print("="*60)
    print(f"Phần chồng lấp tìm thấy ở cuối Chunk 1 và đầu Chunk 2 (Độ dài: {len(overlap_found)} ký tự):")
    print(f"👉 \"{overlap_found}\"")
    print("="*60)

## 5. Kết luận và Đầu ra cuối cùng
Sau khi phân mảnh hoàn tất, mỗi chunk sẽ được đóng gói cùng với thông tin metadata (Ticker, Year, Section, Page, v.v.). 
Dữ liệu đầu ra này sẽ được ghi vào file `data/processed/documents.jsonl` dưới định dạng JSON Lines. Cấu trúc này giúp toàn bộ hệ thống lưu trữ đồng nhất, dễ dàng mở rộng và sẵn sàng cho giai đoạn lập chỉ mục chỉ mục **Indexing** tiếp theo.